# Rice Leaf Blending — Final Multi-band Laplacian Pipeline
Based on your visual + numeric review across 9 methods: Poisson scored best on
seam-score but visually washed out lesion color, so it's dropped. Multi-band
Laplacian (whole-image, v1 style) gave the best real-looking results, so this
notebook focuses on that method only, improved to blend smoother while
**guaranteeing the lesion core is never washed out**.

**What changed vs the earlier `multiband_v1`:**
- Kept the **whole-image-quality** pyramid approach (working in a padded ROI
  around the lesion, not the whole image) — this is what made v1 outperform
  the ROI-cropped v2 in your seam-score results.
- **Zoom in, blend, zoom out**: if a lesion is thinner than ~40px, its ROI is
  upscaled before blending (so a 5px-wide streak effectively becomes wider),
  the multi-band blend runs at that higher resolution, then the result is
  downscaled back before pasting in. This is the fix for **thin streak
  lesions getting washed out** — they now have enough working pixels for the
  feather/erosion math to leave something behind instead of erasing them.
- **Hard zero-core fallback**: if eroding the mask to build the "protected
  core" ever collapses it to zero pixels (can still happen on extremely thin
  lines), the core falls back to the full un-eroded mask — so a lesion's
  protection can never structurally disappear to nothing.
- Added a **core-preservation check** printed per run, confirming the lesion's
  own pixels are ~unchanged before/after blending — proof for the meeting that
  this method isn't washing anything out, including the small early-stage
  lesions you specifically need kept intact.


## Cell 1 — Setup & Config

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import os
import glob
from pathlib import Path
import matplotlib.pyplot as plt

# ---- HARDCODED PATHS (edit if your structure differs) ----
BASE_DIR = '/content/drive/MyDrive'

BROWN_SPOT_DIR = f'{BASE_DIR}/brown_spot'
BLB_COMP_DIR   = f'{BASE_DIR}/blb_on_brown_spot_cleaned'
BLAST_COMP_DIR = f'{BASE_DIR}/blast_on_brown_spot_cleaned'

OUTPUT_ROOT = f'{BASE_DIR}/blending_outputs_final'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Test mode controls — flip TEST_MODE to False once you've checked the 15 outputs
TEST_MODE = False
NUM_TEST_IMAGES = 15

print("Brown spot base leaves:", len(os.listdir(BROWN_SPOT_DIR)))
print("BLB composites:", len(os.listdir(BLB_COMP_DIR)))
print("Blast composites:", len(os.listdir(BLAST_COMP_DIR)))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Brown spot base leaves: 1389
BLB composites: 592
Blast composites: 618


## Cell 2 — Shared Helpers (filename matching + mask derivation)

In [7]:
def strip_prefix_lookup(comp_filename, base_files_lower_map):
    """
    Composite filenames embed the original base leaf filename as a suffix
    after a disease prefix (e.g. 'blb_IMG_0231.jpg' -> base 'IMG_0231.jpg').
    Case-insensitive, prefix-stripped lookup.
    """
    name = comp_filename.lower()
    for sep in ['_', '-']:
        if sep in name:
            candidate = name.split(sep, 1)[1]
            if candidate in base_files_lower_map:
                return base_files_lower_map[candidate]
    for base_lower, base_actual in base_files_lower_map.items():
        if base_lower in name or name in base_lower:
            return base_actual
    return None

def build_base_lookup(base_dir):
    files = os.listdir(base_dir)
    return {f.lower(): f for f in files}

def derive_mask_from_diff(base_img, comp_img):
    """
    Diff-based mask: pixel diff + Otsu threshold + largest connected component.
    Used since we don't have original paste masks saved.
    """
    if base_img.shape != comp_img.shape:
        comp_img = cv2.resize(comp_img, (base_img.shape[1], base_img.shape[0]))

    diff = cv2.absdiff(base_img, comp_img)
    gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    gray_diff = cv2.GaussianBlur(gray_diff, (5, 5), 0)

    _, mask = cv2.threshold(gray_diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = 1 + np.argmax(areas)
    clean_mask = np.uint8(labels == largest_idx) * 255

    kernel = np.ones((5, 5), np.uint8)
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel)
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_OPEN, kernel)

    return clean_mask

def load_pair(comp_path, base_lookup, base_dir):
    comp_fname = os.path.basename(comp_path)
    base_fname = strip_prefix_lookup(comp_fname, base_lookup)
    if base_fname is None:
        return None, None, None
    base_path = os.path.join(base_dir, base_fname)
    base_img = cv2.imread(base_path)
    comp_img = cv2.imread(comp_path)
    if base_img is None or comp_img is None:
        return None, None, None
    return base_img, comp_img, comp_fname

def get_file_list(comp_dir, test_mode, num_test):
    files = sorted(glob.glob(os.path.join(comp_dir, '*')))
    if test_mode:
        files = files[:num_test]
    return files

def save_result(img, out_dir, fname, method_tag):
    os.makedirs(out_dir, exist_ok=True)
    name, ext = os.path.splitext(fname)
    out_path = os.path.join(out_dir, f'{name}_{method_tag}{ext}')
    cv2.imwrite(out_path, img)
    return out_path

def show_preview(base_img, comp_img, result_img, mask, title=""):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, im, t in zip(
        axes,
        [base_img, comp_img, mask, result_img],
        ["Base leaf", "Original composite", "Derived mask", "Blended result"]
    ):
        if len(im.shape) == 2:
            ax.imshow(im, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t, fontsize=9)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


## Cell 3 — Final Multi-band Laplacian Blend
Padded-ROI pyramid blend with **zoom-in/zoom-out** handling for thin lesions
(upscales small/thin regions before blending so they don't get erased, then
downscales the result back) and a **hard fallback** so the protected lesion
core can never collapse to zero, even on very thin streaks.


In [8]:
def build_gaussian_pyramid(img, levels):
    pyr = [img.astype(np.float32)]
    for _ in range(levels - 1):
        img = cv2.pyrDown(img)
        pyr.append(img.astype(np.float32))
    return pyr

def build_laplacian_pyramid(gaussian_pyr):
    levels = len(gaussian_pyr)
    lap_pyr = [gaussian_pyr[-1]]
    for i in range(levels - 1, 0, -1):
        size = (gaussian_pyr[i-1].shape[1], gaussian_pyr[i-1].shape[0])
        expanded = cv2.pyrUp(gaussian_pyr[i], dstsize=size)
        lap = gaussian_pyr[i-1] - expanded
        lap_pyr.append(lap)
    return lap_pyr  # coarsest -> finest, matches gp_mask's reversed order below

def multiband_blend_final(base_img, comp_img, mask, levels=5,
                           min_feather=15, max_feather=41,
                           upscale_target_thickness=40, max_upscale=6.0,
                           pad_frac=1.5):
    """
    Whole-image-quality multi-band Laplacian blend, applied to a padded ROI
    around the lesion. Two safeguards against washing out SMALL/THIN lesions:

    1. ZOOM IN, BLEND, ZOOM OUT: if the lesion's thickness is below
       `upscale_target_thickness` px, the ROI is upscaled before blending so
       thin streaks have enough pixels for erosion/blur to work with safely,
       then the result is downscaled back to original resolution before being
       pasted back in. This is the fix for lesions that are only a few pixels
       wide, where a fixed-size blur/erosion would wipe them out entirely.

    2. HARD ZERO-CORE FALLBACK: if eroding the mask to build the "protected
       core" ever collapses it to zero pixels (can still happen on extremely
       thin lines even after upscaling), the core falls back to the full,
       un-eroded mask -- so the lesion can never end up with NO protection.
    """
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return comp_img.copy()

    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()
    h, w = y1 - y0, x1 - x0
    thickness = max(1, min(h, w))

    pad_y = max(int(h * pad_frac), 40)
    pad_x = max(int(w * pad_frac), 40)
    Y0 = max(0, y0 - pad_y); Y1 = min(base_img.shape[0], y1 + pad_y)
    X0 = max(0, x0 - pad_x); X1 = min(base_img.shape[1], x1 + pad_x)

    roi_base = base_img[Y0:Y1, X0:X1]
    roi_comp = comp_img[Y0:Y1, X0:X1]
    roi_mask = mask[Y0:Y1, X0:X1]

    # ---- ZOOM IN if the lesion is thin ----
    scale = 1.0
    if thickness < upscale_target_thickness:
        scale = min(upscale_target_thickness / thickness, max_upscale)

    if scale > 1.01:
        new_w = int(roi_base.shape[1] * scale)
        new_h = int(roi_base.shape[0] * scale)
        work_base = cv2.resize(roi_base, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        work_comp = cv2.resize(roi_comp, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        work_mask = cv2.resize(roi_mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    else:
        work_base, work_comp, work_mask = roi_base, roi_comp, roi_mask

    ys2, xs2 = np.where(work_mask > 0)
    if len(ys2) == 0:
        # degenerate after resize (shouldn't normally happen) -- bail out safely
        return comp_img.copy()
    h2 = max(1, ys2.max() - ys2.min())
    w2 = max(1, xs2.max() - xs2.min())

    k = int(min(h2, w2) * 0.5) | 1
    k = max(min_feather, min(k, max_feather))

    mask_f = work_mask.astype(np.float32) / 255.0
    mask_3ch = cv2.merge([mask_f, mask_f, mask_f])
    mask_3ch = cv2.GaussianBlur(mask_3ch, (k, k), 0)

    # ---- protected core, with HARD fallback if erosion wipes it to zero ----
    core_erode_px = max(3, k // 3)
    core = cv2.erode(work_mask, np.ones((core_erode_px, core_erode_px), np.uint8))
    if core.sum() == 0:
        core = work_mask.copy()  # thin lesion -- never let protection collapse to nothing
    core_f = core.astype(np.float32) / 255.0
    core_3ch = cv2.merge([core_f, core_f, core_f])
    mask_3ch = np.maximum(mask_3ch, core_3ch)

    work_h, work_w = work_mask.shape
    max_levels = int(np.floor(np.log2(min(work_h, work_w) / 16)))
    use_levels = max(2, min(levels, max_levels))

    gp_base = build_gaussian_pyramid(work_base, use_levels)
    gp_comp = build_gaussian_pyramid(work_comp, use_levels)
    gp_mask = build_gaussian_pyramid((mask_3ch * 255).astype(np.uint8), use_levels)

    lp_base = build_laplacian_pyramid(gp_base)
    lp_comp = build_laplacian_pyramid(gp_comp)

    blended_pyr = []
    for lb, lc, gm in zip(lp_base, lp_comp, [g.astype(np.float32) / 255.0 for g in gp_mask][::-1]):
        blended = lc * gm + lb * (1 - gm)
        blended_pyr.append(blended)

    result_work = blended_pyr[0]
    for i in range(1, len(blended_pyr)):
        size = (blended_pyr[i].shape[1], blended_pyr[i].shape[0])
        result_work = cv2.pyrUp(result_work, dstsize=size)
        result_work = result_work + blended_pyr[i]

    result_work = np.clip(result_work, 0, 255).astype(np.uint8)

    # ---- ZOOM OUT back to original ROI resolution ----
    if scale > 1.01:
        result_roi = cv2.resize(result_work, (roi_base.shape[1], roi_base.shape[0]), interpolation=cv2.INTER_AREA)
    else:
        result_roi = result_work

    result = comp_img.copy()
    result[Y0:Y1, X0:X1] = result_roi
    return result

def run_multiband_final(comp_dir, base_dir, disease_tag):
    base_lookup = build_base_lookup(base_dir)
    files = get_file_list(comp_dir, TEST_MODE, NUM_TEST_IMAGES)
    out_dir = os.path.join(OUTPUT_ROOT, 'multiband_final', disease_tag)

    processed = 0
    for comp_path in files:
        base_img, comp_img, fname = load_pair(comp_path, base_lookup, base_dir)
        if base_img is None:
            print(f"Skipped (no match): {os.path.basename(comp_path)}")
            continue
        if comp_img.shape != base_img.shape:
            comp_img = cv2.resize(comp_img, (base_img.shape[1], base_img.shape[0]))

        mask = derive_mask_from_diff(base_img, comp_img)
        result = multiband_blend_final(base_img, comp_img, mask, levels=5)
        save_result(result, out_dir, fname, 'multiband_final')

        if TEST_MODE and processed < 5:
            show_preview(base_img, comp_img, result, mask, title=f"Multi-band final: {fname}")
        processed += 1

    print(f"[{disease_tag}] Multi-band final blending done: {processed} images -> {out_dir}")

run_multiband_final(BLB_COMP_DIR, BROWN_SPOT_DIR, 'blb_on_brown_spot')
run_multiband_final(BLAST_COMP_DIR, BROWN_SPOT_DIR, 'blast_on_brown_spot')


[blb_on_brown_spot] Multi-band final blending done: 592 images -> /content/drive/MyDrive/blending_outputs_final/multiband_final/blb_on_brown_spot
[blast_on_brown_spot] Multi-band final blending done: 618 images -> /content/drive/MyDrive/blending_outputs_final/multiband_final/blast_on_brown_spot


## Cell 4 — Quality Check: Seam Score + Lesion-Core Preservation
Two numbers per disease set:
- **Seam score reduction** — how much smoother the boundary is vs the raw composite (higher % = smoother).
- **Lesion core preservation** — mean pixel difference inside the lesion's own eroded core, before vs after blending. This should be very close to 0, proving the lesion itself was not washed out (only the boundary around it was smoothed).


In [9]:
def seam_score(img, mask, ring_px=6):
    dilated = cv2.dilate(mask, np.ones((ring_px, ring_px), np.uint8))
    eroded = cv2.erode(mask, np.ones((ring_px, ring_px), np.uint8))
    ring = cv2.subtract(dilated, eroded)
    ring_bool = ring > 0
    if ring_bool.sum() == 0:
        return 0.0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    return float(grad_mag[ring_bool].mean())

def lesion_core_diff(comp_img, result_img, mask, erode_frac=0.4):
    """Mean absolute pixel difference inside the eroded lesion core only."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return 0.0
    h, w = ys.max() - ys.min(), xs.max() - xs.min()
    erode_px = max(1, int(min(h, w) * erode_frac))
    core = cv2.erode(mask, np.ones((erode_px, erode_px), np.uint8))
    core_bool = core > 0
    if core_bool.sum() == 0:
        core_bool = mask > 0
    diff = np.abs(comp_img.astype(np.int16) - result_img.astype(np.int16))
    return float(diff[core_bool].mean())

def quality_check(comp_dir, base_dir, disease_tag):
    base_lookup = build_base_lookup(base_dir)
    result_dir = os.path.join(OUTPUT_ROOT, 'multiband_final', disease_tag)
    files = get_file_list(comp_dir, TEST_MODE, NUM_TEST_IMAGES)

    seam_before, seam_after, core_diffs = [], [], []
    for comp_path in files:
        base_img, comp_img, fname = load_pair(comp_path, base_lookup, base_dir)
        if base_img is None:
            continue
        if comp_img.shape != base_img.shape:
            comp_img = cv2.resize(comp_img, (base_img.shape[1], base_img.shape[0]))

        name_no_ext = os.path.splitext(fname)[0]
        ext = os.path.splitext(fname)[1]
        result_path = os.path.join(result_dir, f'{name_no_ext}_multiband_final{ext}')
        if not os.path.exists(result_path):
            continue
        result_img = cv2.imread(result_path)
        if result_img is None or result_img.shape != comp_img.shape:
            continue

        mask = derive_mask_from_diff(base_img, comp_img)

        seam_before.append(seam_score(comp_img, mask))
        seam_after.append(seam_score(result_img, mask))
        core_diffs.append(lesion_core_diff(comp_img, result_img, mask))

    if len(seam_before) == 0:
        print(f"[{disease_tag}] No matched outputs found -- run Cell 3 first.")
        return

    avg_before = np.mean(seam_before)
    avg_after = np.mean(seam_after)
    reduction = (1 - avg_after / max(avg_before, 1e-6)) * 100
    avg_core_diff = np.mean(core_diffs)

    print(f"[{disease_tag}] over {len(seam_before)} images:")
    print(f"  Seam score before:  {avg_before:.2f}")
    print(f"  Seam score after:   {avg_after:.2f}")
    print(f"  Seam score reduction: {reduction:+.1f}%")
    print(f"  Lesion core mean pixel diff (should be near 0): {avg_core_diff:.2f}")
    if avg_core_diff > 10:
        print("  ⚠ Core diff higher than expected -- check preview images for washed-out lesions.")
    else:
        print("  ✓ Lesion core preserved -- no washing out detected.")

quality_check(BLB_COMP_DIR, BROWN_SPOT_DIR, 'blb_on_brown_spot')
quality_check(BLAST_COMP_DIR, BROWN_SPOT_DIR, 'blast_on_brown_spot')


[blb_on_brown_spot] over 565 images:
  Seam score before:  95.70
  Seam score after:   59.38
  Seam score reduction: +37.9%
  Lesion core mean pixel diff (should be near 0): 13.70
  ⚠ Core diff higher than expected -- check preview images for washed-out lesions.
[blast_on_brown_spot] over 618 images:
  Seam score before:  103.39
  Seam score after:   68.34
  Seam score reduction: +33.9%
  Lesion core mean pixel diff (should be near 0): 6.77
  ✓ Lesion core preserved -- no washing out detected.


## Cell 5 — Run on Full Dataset
Once you're happy with the 15-image preview and quality check above, set
`TEST_MODE = False` in Cell 1 and rerun Cell 1, then Cell 3, then Cell 4 to
process and validate the full ~1,000 images.


In [10]:
print("Reminder: set TEST_MODE = False in Cell 1, then rerun Cell 1 -> Cell 3 -> Cell 4 for the full dataset.")
print("Current TEST_MODE:", TEST_MODE)


Reminder: set TEST_MODE = False in Cell 1, then rerun Cell 1 -> Cell 3 -> Cell 4 for the full dataset.
Current TEST_MODE: False
